# Build Structured JSON Extractor

A **Structured JSON Extractor** for LLMs takes unstructured documents (PDFs, OCR text, invoices, contracts) and converts them into a predefined JSON schema while validating the output against that schema.

## Architecture

```text
Invoice / Contract (PDF, Image, Text)
                │
                ▼
         OCR (if scanned)
                │
                ▼
        Text Preprocessing
                │
                ▼
      LLM + Structured Prompt
                │
                ▼
        JSON Extraction
                │
                ▼
      JSON Schema Validation
                │
      ┌─────────┴─────────┐
      │                   │
   Valid JSON         Validation Error
      │                   │
      ▼                   ▼
 Save to DB         Retry / Repair Prompt
```

---

## Approach 1 - Using Json Schema
## Example 1: Invoice Extraction Schema

```json
{
  "invoice_number": "string",
  "invoice_date": "YYYY-MM-DD",
  "due_date": "YYYY-MM-DD",
  "vendor": {
    "name": "string",
    "address": "string",
    "gst_number": "string"
  },
  "customer": {
    "name": "string",
    "address": "string"
  },
  "currency": "string",
  "subtotal": "number",
  "tax": "number",
  "total": "number",
  "line_items": [
    {
      "description": "string",
      "quantity": "number",
      "unit_price": "number",
      "amount": "number"
    }
  ]
}
```

---

## Example 2: Contract Extraction Schema

```json
{
  "contract_title": "string",
  "effective_date": "YYYY-MM-DD",
  "expiration_date": "YYYY-MM-DD",
  "parties": [
    {
      "name": "string",
      "role": "string"
    }
  ],
  "payment_terms": "string",
  "termination_clause": "string",
  "governing_law": "string",
  "confidentiality": "string"
}
```

---

## Prompt Template

```text
You are an information extraction assistant.

Extract the required information from the document.

Rules:
- Return ONLY valid JSON.
- Follow the schema exactly.
- Use null if information is missing.
- Do not invent values.
- Dates must use YYYY-MM-DD.
- Numbers must be numeric.
```

Attach the JSON schema after the instructions.

---

## Python Example (OpenAI SDK)

```python
from openai import OpenAI
import json

client = OpenAI()

schema = {
    "type": "object",
    "properties": {
        "invoice_number": {"type": "string"},
        "invoice_date": {"type": "string"},
        "vendor": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "gst_number": {"type": "string"}
            },
            "required": ["name"]
        },
        "total": {"type": "number"}
    },
    "required": [
        "invoice_number",
        "invoice_date",
        "vendor",
        "total"
    ]
}

text = open("invoice.txt").read()

response = client.responses.create(
    model="gpt-5.5",
    input=[
        {
            "role": "system",
            "content": "Extract structured invoice data."
        },
        {
            "role": "user",
            "content": text
        }
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "invoice_schema",
            "schema": schema,
            "strict": True
        }
    }
)

print(response.output_text)
```

---

## Schema Validation

Validate the LLM output before storing or using it.

```python
from jsonschema import validate, ValidationError
import json

data = json.loads(response.output_text)

try:
    validate(instance=data, schema=schema)
    print("Valid JSON")
except ValidationError as e:
    print("Validation failed:", e.message)
```

---

## Example Input (Invoice)

```text
Invoice #: INV-2045

Vendor:
ABC Technologies Pvt Ltd

Invoice Date: 2025-06-12

Customer:
XYZ Ltd

Laptop           2 × 50000 = 100000
Mouse            5 × 800 = 4000

GST 18%: 18720

Total: 122720 INR
```

### Extracted Output

```json
{
  "invoice_number": "INV-2045",
  "invoice_date": "2025-06-12",
  "due_date": null,
  "vendor": {
    "name": "ABC Technologies Pvt Ltd",
    "address": null,
    "gst_number": null
  },
  "customer": {
    "name": "XYZ Ltd",
    "address": null
  },
  "currency": "INR",
  "subtotal": 104000,
  "tax": 18720,
  "total": 122720,
  "line_items": [
    {
      "description": "Laptop",
      "quantity": 2,
      "unit_price": 50000,
      "amount": 100000
    },
    {
      "description": "Mouse",
      "quantity": 5,
      "unit_price": 800,
      "amount": 4000
    }
  ]
}
```

---

## Example Output (Contract)

```json
{
  "contract_title": "Software Development Agreement",
  "effective_date": "2025-01-01",
  "expiration_date": "2026-01-01",
  "parties": [
    {
      "name": "ABC Technologies",
      "role": "Vendor"
    },
    {
      "name": "XYZ Corporation",
      "role": "Client"
    }
  ],
  "payment_terms": "Net 30 days",
  "termination_clause": "30 days written notice",
  "governing_law": "India",
  "confidentiality": "Mutual NDA applies"
}
```

## Best Practices

* Use **strict JSON schema enforcement** so the model cannot return fields outside the defined schema.
* Instruct the model to return **`null` for missing values** instead of guessing.
* Normalize dates (`YYYY-MM-DD`), currencies (ISO 4217 codes where possible), and numeric values.
* Validate every response with a JSON Schema validator before downstream processing.
* If validation fails, automatically retry with a repair prompt that includes the validation error and asks the model to return corrected JSON only.
* For scanned PDFs, perform OCR first (for example, with Tesseract or cloud OCR services), then pass the extracted text to the LLM.
* Add confidence scores or source spans for extracted entities if auditability is important, especially in finance or legal workflows.


## Approach 2 - Using classes

In [2]:
!python structured_extractor_openai.py sample_invoice.txt

{
  "document_type": "invoice",
  "confidence": 1.0,
  "invoice": {
    "document_type": "invoice",
    "invoice_number": "INV-20458",
    "issue_date": "2026-06-12",
    "due_date": "2026-07-12",
    "vendor": {
      "name": "Acme Industrial Supply Co.",
      "role": "Vendor",
      "address": "442 Foundry Lane, Detroit, MI 48201",
      "email": "billing@acmeindustrial.com",
      "phone": "(313) 555-0199",
      "tax_id": "38-1234567"
    },
    "customer": {
      "name": "Northwind Manufacturing LLC",
      "role": "Customer",
      "address": "900 Assembly Row, Toledo, OH 43604",
      "email": null,
      "phone": null,
      "tax_id": null
    },
    "line_items": [
      {
        "description": "Hydraulic press gaskets (set of 12)",
        "quantity": 4.0,
        "unit_price": 85.0,
        "line_total": 340.0
      },
      {
        "description": "Stainless steel fasteners (box of 500)",
        "quantity": 10.0,
        "unit_price": 22.5,
        "line_total": 225.0


INFO: HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
